# **Classification: Gradient Boosting Classifier (GBC)**

## **Justification of Preprocessing Strategy**

### **Scale Invariance**
The Gradient Boosting Classifier is a powerful ensemble of Decision Trees. Since each split in these base trees is determined by finding a discrete threshold for a single feature, the algorithm is mathematically invariant to the scale of the inputs. Whether the data is standardized, normalized, or left in its raw form, the resulting decision boundaries remain completely identical. Consequently, we will use the **Original, Unscaled Data** to preserve computational efficiency and clinical interpretability.

### **The Boosting Philosophy: Why We Do Not Use the Decision Tree Champion**
Unlike Bagging (which relies on deep, complex trees to capture variance), Gradient Boosting builds trees *sequentially*. Each new tree attempts to minimize the residual errors of the previous ensemble using a gradient-descent approach. 
Because of this sequential learning, GBC requires **Weak Learners** (shallow trees, typically with a `max_depth` of 3 to 5). If we were to inject our optimized Decision Tree champion (`max_depth=10`), the very first tree would severely overfit, leaving subsequent trees to merely amplify noise. Therefore, we let the algorithm build its own shallow trees from scratch.

---

## **Experiment Design & Computational Efficiency**

We defined a tournament of 3 optimization levels to identify the most robust configuration. Because Gradient Boosting is highly prone to overfitting if the `learning_rate` is too high or the trees get too deep, we strictly log **both Train and Test metrics** across all runs to visually monitor the learning gap. 

Furthermore, due to the sequential nature of Boosting, training on a 100,000-row dataset is computationally expensive. We intentionally narrowed our Search spaces to prioritize efficiency and prevent unfeasible execution times:

* **Baseline**: Utilizing Scikit-Learn's default parameters (e.g., `n_estimators=100`, `max_depth=3`, `learning_rate=0.1`) to establish a pure, unconstrained performance reference.
* **GridSearchCV**: A highly targeted 3-fold cross-validated search. We drastically reduced the grid to exclusively test the boundaries of ensemble size (`n_estimators`), step size (`learning_rate`), tree complexity (`max_depth` strictly between 3 and 5), and data starvation (`subsample`).
* **Optuna Optimization**: Bayesian optimization utilizing a fixed budget of 12 trials to efficiently explore the continuous range of these critical hyperparameters, aiming to surgically maximize generalizable Recall.

In [1]:
import pandas as pd
import numpy as np
import time
import mlflow
import optuna
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import recall_score, accuracy_score, f1_score

# MLflow Configuration
mlflow.set_tracking_uri("sqlite:///C:/Users/Tiago Silva/Uni/OneDrive - Universidade Portucalense/Ambiente de Trabalho/Uni/3ano2sem/LAD/Grupo5_ProjetoLAD_Parte2/TrabalhoLAD/models/mlflow.db")
mlflow.set_experiment("Classification_GradientBoosting")

# Data Loading and Preparation
df = pd.read_csv("C:\\Users\\Tiago Silva\\Uni\\OneDrive - Universidade Portucalense\\Ambiente de Trabalho\\Uni\\3ano2sem\\LAD\\Grupo5_ProjetoLAD_Parte2\\TrabalhoLAD\\data\\diabetes_dataset_new_variables.csv")

categorical_cols = [
    'gender', 'ethnicity', 'smoking_status', 'education_level',
    'employment_status', 'age_groups', 'weight_status', 'income_level'
]

# Apply One-Hot Encoding
df_final = pd.get_dummies(df, columns=categorical_cols, drop_first=True)

X = df_final.drop(["diagnosed_diabetes", "diabetes_stage"], axis=1)
y = df_final['diagnosed_diabetes']

# Split data (80/20) maintaining class proportion
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

def log_classification_metrics(model, X_tr, y_tr, X_te, y_te, duration):
    """Logs both Train and Test metrics to explicitly monitor the Overfitting Gap"""
    y_tr_pred = model.predict(X_tr)
    y_te_pred = model.predict(X_te)
    
    # Train Partition Metrics
    mlflow.log_metric("recall_train", recall_score(y_tr, y_tr_pred))
    mlflow.log_metric("accuracy_train", accuracy_score(y_tr, y_tr_pred))
    mlflow.log_metric("f1_train", f1_score(y_tr, y_tr_pred))
    
    # Test Partition Metrics
    mlflow.log_metric("recall_test", recall_score(y_te, y_te_pred))
    mlflow.log_metric("accuracy_test", accuracy_score(y_te, y_te_pred))
    mlflow.log_metric("f1_test", f1_score(y_te, y_te_pred))
    
    mlflow.log_metric("fit_time", duration)

# ---------------------------------------------------------
# RUN 1: BASELINE 
# ---------------------------------------------------------
with mlflow.start_run(run_name="GBC_Baseline_Defaults"):
    # Calling the classifier with absolute defaults (max_depth=3, learning_rate=0.1)
    gb_base = GradientBoostingClassifier(random_state=42)
    
    start_time = time.time()
    gb_base.fit(X_train, y_train)
    duration = time.time() - start_time
    
    mlflow.log_params(gb_base.get_params())
    mlflow.log_param("optimization", "none_default")
    
    log_classification_metrics(gb_base, X_train, y_train, X_test, y_test, duration)

# ---------------------------------------------------------
# RUN 2: GRIDSEARCHCV 
# ---------------------------------------------------------
with mlflow.start_run(run_name="GBC_GridSearch"):
    # Target grid reduced drastically to prevent computational timeouts
    param_grid = {
        'n_estimators': [50, 150],          
        'learning_rate': [0.01, 0.1],         
        'max_depth': [3, 5],                  
        'subsample': [0.8, 1.0]               
    }
    
    grid = GridSearchCV(
        GradientBoostingClassifier(random_state=42),
        param_grid, cv=3, scoring='recall', n_jobs=-1
    )
    
    start_time = time.time()
    grid.fit(X_train, y_train)
    duration = time.time() - start_time
    
    best_gbc_grid = grid.best_estimator_
    
    mlflow.log_params(grid.best_params_)
    mlflow.log_param("optimization", "GridSearchCV")
    
    log_classification_metrics(best_gbc_grid, X_train, y_train, X_test, y_test, duration)

# ---------------------------------------------------------
# RUN 3: OPTUNA 
# ---------------------------------------------------------
def objective(trial):
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 50, 200),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.2, log=True),
        "max_depth": trial.suggest_int("max_depth", 2, 5), # Kept safely constrained
        "subsample": trial.suggest_float("subsample", 0.7, 1.0)
    }
    
    model = GradientBoostingClassifier(random_state=42, **params)
    # Cross-validation focusing entirely on the Train set to avoid leakage
    score = cross_val_score(model, X_train, y_train, cv=3, scoring='recall', n_jobs=-1).mean()
    return score

with mlflow.start_run(run_name="GBC_Optuna"):
    study = optuna.create_study(direction="maximize")
    start_time = time.time()
    # Fixed budget of 12 trials ensures predictable execution time
    study.optimize(objective, n_trials=12) 
    duration = time.time() - start_time
    
    # Train final champion model
    best_gbc_opt = GradientBoostingClassifier(**study.best_params, random_state=42)
    best_gbc_opt.fit(X_train, y_train)
    
    mlflow.log_params(study.best_params)
    mlflow.log_param("optimization", "optuna")
    
    log_classification_metrics(best_gbc_opt, X_train, y_train, X_test, y_test, duration)

[I 2026-05-21 10:25:02,519] A new study created in memory with name: no-name-f61fa91b-390e-4a23-b54f-e7284bcfe543
[I 2026-05-21 10:25:23,520] Trial 0 finished with value: 0.8689736793445423 and parameters: {'n_estimators': 178, 'learning_rate': 0.027766648558539157, 'max_depth': 2, 'subsample': 0.7227460764108233}. Best is trial 0 with value: 0.8689736793445423.
[I 2026-05-21 10:25:45,918] Trial 1 finished with value: 0.8690153486155384 and parameters: {'n_estimators': 114, 'learning_rate': 0.050246156839983444, 'max_depth': 3, 'subsample': 0.8765959308968054}. Best is trial 1 with value: 0.8690153486155384.
[I 2026-05-21 10:25:54,953] Trial 2 finished with value: 0.8689736793445423 and parameters: {'n_estimators': 82, 'learning_rate': 0.014361729337337823, 'max_depth': 2, 'subsample': 0.7829246401749822}. Best is trial 1 with value: 0.8690153486155384.
[I 2026-05-21 10:26:02,077] Trial 3 finished with value: 0.8689736793445423 and parameters: {'n_estimators': 53, 'learning_rate': 0.01

## Runs Summary

| Run | Accuracy (Train) | Accuracy (Test) | F1 (Train) | F1 (Test) | Recall (Train) | Recall (Test) | Fit Time |
|---|---:|---:|---:|---:|---:|---:|---:|
| GBC_Baseline_Defaults | 0.92155 | 0.91985 | 0.9300490415 | 0.9284343051 | 0.8692445519 | 0.86650 | 27.41s |
| GBC_GridSearch | 0.9234875 | 0.91950 | 0.9318958132 | 0.9281891169 | 0.8724946873 | 0.8670833333 | 228.37s |
| GBC_Optuna | 0.9330125 | 0.91605 | 0.9408949034 | 0.9254870634 | 0.8886828618 | 0.8689166667 | 356.72s |

### Additional logged parameters
- `random_state = 42` where set in code
- Hyperparameters optimized: `n_estimators`, `learning_rate`, `max_depth`, `subsample`

## Winner Run Selection (Priority Elimination Framework)

### New Policy (effective immediately)
A run is only eligible to win if it does **not** show evidence of overfitting or underfitting. Before applying the Recall/F1/fit_time decision rules, we require the **Recall** and **F1** Train→Test gaps (Test − Train) to remain within ±0.5 percentage points (|gap| ≤ 0.005) to consider a run as generalizing. If a run fails this check it is disqualified regardless of metric rank.

### Selection Criteria (priority order)
1. **Generalization filter (mandatory):** Recall and F1 gaps within ±0.5 percentage points. Disqualified runs are removed from consideration.
2. **Priority 1 (70%): Highest Recall (Test)** — clinical priority: maximize detection of positive diabetes cases.
3. **Priority 2 (30%): Highest F1-Score (Test)** — used when Recall ties or differs by <0.5% among remaining candidates.
4. **Accuracy is visible but ignored** — shown for reference only; not used in selection.
5. **Tiebreaker: Lowest Fit Time** — if Recall and F1 remain tied.

### Generalization Check (Test − Train)
- **GBC_Baseline_Defaults:** Recall gap = 0.86650 − 0.8692445519 = **−0.27pp** → PASS. F1 gap = 0.9284343051 − 0.9300490415 = **−0.16pp** → PASS.
- **GBC_GridSearch:** Recall gap = 0.8670833333 − 0.8724946873 = **−0.54pp** → FAIL. F1 gap = 0.9281891169 − 0.9318958132 = **−0.37pp** → PASS.
- **GBC_Optuna:** Recall gap = 0.8689166667 − 0.8886828618 = **−1.98pp** → FAIL. F1 gap = 0.9254870634 − 0.9408949034 = **−1.54pp** → FAIL.

### Step-by-Step Elimination
**Step 1 — Apply the generalization filter**
- Passing run: **GBC_Baseline_Defaults**.
- Disqualified runs: GBC_GridSearch, GBC_Optuna.

**Step 2 — Among eligible runs, compare Test Recall (Priority 1 — 70%)**
- Only candidate: **GBC_Baseline_Defaults**.

**Step 3 — Verify F1 (Priority 2 — 30%)**
- Not required; only one candidate remains.

**Step 4 — Fit Time tiebreaker**
- Not required.

### Final Decision
**Winner: GBC_Baseline_Defaults**

**Justification:** Only `GBC_Baseline_Defaults` satisfies the mandatory generalization filter. `GBC_GridSearch` fails the Recall gap threshold by a small margin, and `GBC_Optuna` shows clear overfitting with large Train→Test gaps on both Recall and F1. Therefore, the baseline is the only valid winner under the new policy.

## Winner Hyperparameters

| Parameter | Value |
|---|---|
| **n_estimators** | 100 |
| **learning_rate** | 0.1 |
| **max_depth** | 3 |
| **subsample** | 1.0 |
| **random_state** | 42 |

## Overfitting / Underfitting Diagnosis
- `GBC_Baseline_Defaults` shows small Train→Test gaps and passes the generalization rule, so it is the only run that does **not** show disqualifying overfitting or underfitting.
- `GBC_GridSearch` shows a small but real Recall gap beyond the ±0.5pp threshold, so it is disqualified.
- `GBC_Optuna` shows clear overfitting because both Recall and F1 gaps are well beyond the threshold.